# Approximation of $G_{rs}$ versus the frequency cutoff

For **one fixed sample** and **one fixed bandwidth pair** $(h_r,h_s)$, compare the analytic Gaussian-KDE inner product (dashed horizontal line) with the existing Fourier approximation (solid line). Only $\Omega$ varies along the curve; **$K$ stays fixed**.

Here $K$ counts trapezoidal **intervals**, so the helper receives `frequency_points=K+1`, with $\omega_k=k\Omega/K$ for $k=0,\ldots,K$. The reference is

$$G_{rs}=\frac{1}{n^2}\sum_{i,j=1}^n\phi_{\sqrt{h_r^2+h_s^2}}(X_i-X_j).$$

Source: [thesis, Equation `eq:beta2_exact_formula02` and “Numerical Approximation”](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex), [analytic implementation](../kde_tools/density.py), and [frequency-quadrature implementation](../kde_tools/fourier/similarities.py). The latter uses the [composite trapezoidal rule](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.trapezoid.html).

The settings below are editable. PDFs are saved automatically under `../outputs/pdfs/g_rs_approximation_vs_omega/`; the PNG preview, sample, settings, and curve data are saved separately under `../outputs/`.


In [ ]:
from pathlib import Path
import json
import sys

code_directory = next(
    (candidate.resolve()
     for ancestor in (Path.cwd(), *Path.cwd().parents)
     for candidate in (ancestor, ancestor / "Code" / "Code_kde")
     if (candidate / "kde_tools" / "pdf_export.py").is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from inside the MASTERARBEIT project.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

import matplotlib.pyplot as plt
import numpy as np

from kde_tools.density import exact_gaussian_kde_l2_gram
from kde_tools.fourier import FOURIER_L2_METHOD_VERSION, gaussian_kde_l2_gram_fourier
from kde_tools.pdf_export import enable_automatic_pdf_saving

notebook_name = "g_rs_approximation_vs_omega"
pdf_output_directory = enable_automatic_pdf_saving(notebook_name)
data_directory = code_directory / "outputs" / "data" / notebook_name
image_directory = code_directory / "outputs" / "images" / notebook_name
data_directory.mkdir(parents=True, exist_ok=True)
image_directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# Fixed sample: drawn once in the calculation cell, then reused for every Omega.
n = 200
seed = 20261024

# One fixed pair of Gaussian KDE bandwidths.
h_r = 0.2
h_s = 0.4

# K is the number of trapezoidal intervals, not the number of nodes.
K = 32
omega_min = 0.05
omega_max = 80.0
omega_points = 641  # Number of cutoff values shown on the horizontal axis.


In [ ]:
if not isinstance(n, (int, np.integer)) or isinstance(n, bool) or n < 1:
    raise ValueError("n must be a positive integer.")
if not isinstance(K, (int, np.integer)) or isinstance(K, bool) or K < 1:
    raise ValueError("K must be a positive integer counting intervals.")
if (not isinstance(omega_points, (int, np.integer))
        or isinstance(omega_points, bool) or omega_points < 2):
    raise ValueError("omega_points must be an integer of at least two.")
if not (np.isfinite(omega_min) and np.isfinite(omega_max)
        and 0.0 < omega_min < omega_max):
    raise ValueError("Require finite cutoffs with 0 < omega_min < omega_max.")

samples = np.random.default_rng(seed).normal(size=n)
bandwidths = np.array([h_r, h_s], dtype=float)
if not np.all(np.isfinite(bandwidths)) or np.any(bandwidths <= 0):
    raise ValueError("Both bandwidths must be finite and strictly positive.")

# Analytic reference for this sample and this bandwidth pair.
exact_value = float(exact_gaussian_kde_l2_gram(samples, bandwidths)[0, 1])
omega_values = np.linspace(omega_min, omega_max, omega_points)

# The same K+1 nodes are used at every cutoff. Their spacing is Omega/K.
approximation_values = np.array([
    gaussian_kde_l2_gram_fourier(
        samples,
        bandwidths,
        frequency_cutoff=float(omega),
        frequency_points=K + 1,
    )[0, 1]
    for omega in omega_values
])
absolute_errors = np.abs(approximation_values - exact_value)

np.savetxt(data_directory / "sample.csv", samples, delimiter=",", header="x", comments="")
np.savetxt(
    data_directory / "g_rs_vs_omega.csv",
    np.column_stack((omega_values, approximation_values,
                     np.full_like(omega_values, exact_value), absolute_errors)),
    delimiter=",",
    header="omega,approximation,exact,absolute_error",
    comments="",
)
settings = {
    "sample_distribution": "standard_normal", "n": int(n), "seed": int(seed),
    "h_r": float(h_r), "h_s": float(h_s), "K": int(K),
    "frequency_points": int(K + 1),
    "omega_min": float(omega_min), "omega_max": float(omega_max),
    "omega_points": int(omega_points), "exact_value": exact_value,
    "method_version": FOURIER_L2_METHOD_VERSION,
}
(data_directory / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print(f"Exact G_rs = {exact_value:.12g}")
print(f"Fixed K = {K} intervals ({K + 1} nodes at every cutoff).")


In [ ]:
with plt.rc_context({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
}):
    fig, ax = plt.subplots(figsize=(6.2, 3.7), layout="constrained")
    ax.plot(
        omega_values, approximation_values,
        color="#2563a6", linewidth=2.0,
        label=rf"Approximation ($K={K}$)",
    )
    ax.axhline(
        exact_value, color="#222222", linestyle="--", linewidth=1.5,
        label=r"Exact $G_{rs}$", zorder=3,
    )
    ax.set_xlabel(r"Frequency cutoff $\Omega$")
    ax.set_ylabel(r"$G_{rs}$")
    ax.set_title(
        rf"$G_{{rs}}$ approximation at fixed $K={K}$" + "\n"
        + rf"$n={n},\quad h_r={h_r:g},\quad h_s={h_s:g}$",
        fontsize=11, pad=10,
    )
    ax.set_xlim(0.0, omega_max)
    ax.set_ylim(bottom=0.0)
    ax.grid(axis="y", color="#e6e6e6", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.legend(loc="upper left", frameon=False)
    png_path = image_directory / "g_rs_vs_omega.png"
    fig.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.show()  # Shared exporter automatically saves this figure as PDF.


At fixed $K$, increasing $\Omega$ also increases the node spacing $\Delta\omega=\Omega/K$. This experiment therefore changes both the truncated interval and its mesh; it is not a convergence test with fixed node spacing. That observation follows directly from the grid definition above. The saved CSV contains the actual error for each displayed cutoff.


## Compare several fixed values of K

Overlay one approximation curve for each value in `K_values`, using exactly the same `samples`, `bandwidths`, `omega_values`, and analytic reference as above. Each curve holds its own K fixed while Omega varies. The original single-K figure and its exports are retained.


In [ ]:
# Edit this list to compare other numbers of trapezoidal intervals.
K_values = [16, 32, 64, 128]
if (not K_values or any(
        not isinstance(value, (int, np.integer))
        or isinstance(value, bool) or value < 1
        for value in K_values)):
    raise ValueError("K_values must contain positive integers counting intervals.")
if len(set(K_values)) != len(K_values):
    raise ValueError("K_values must not contain duplicates.")

approximation_curves = {}
for current_K in K_values:
    approximation_curves[current_K] = np.array([
        gaussian_kde_l2_gram_fourier(
            samples,
            bandwidths,
            frequency_cutoff=float(omega),
            frequency_points=int(current_K) + 1,
        )[0, 1]
        for omega in omega_values
    ])

np.savetxt(
    data_directory / "g_rs_vs_omega_multiple_k.csv",
    np.vstack([
        np.column_stack((
            np.full_like(omega_values, current_K),
            omega_values,
            approximation_curves[current_K],
            np.full_like(omega_values, exact_value),
            np.abs(approximation_curves[current_K] - exact_value),
        ))
        for current_K in K_values
    ]),
    delimiter=",",
    header="K,omega,approximation,exact,absolute_error",
    comments="",
)
multiple_k_settings = {
    key: value for key, value in settings.items()
    if key not in ("K", "frequency_points")
}
multiple_k_settings["K_values"] = [int(value) for value in K_values]
multiple_k_settings["frequency_points_by_K"] = {
    str(value): int(value) + 1 for value in K_values
}
(data_directory / "settings_multiple_k.json").write_text(
    json.dumps(multiple_k_settings, indent=2) + "\n"
)
print(f"Same fixed sample: n={len(samples)}, exact G_rs={exact_value:.12g}")
print(f"K values: {K_values}")


In [ ]:
with plt.rc_context({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
}):
    fig_multiple_k, ax_multiple_k = plt.subplots(
        figsize=(6.5, 4.0), layout="constrained"
    )
    curve_colors = ["#d55e00", "#2563a6", "#009e73", "#a455a0"]
    curve_styles = ["-", "-", "-.", ":"]
    for index, current_K in enumerate(K_values):
        ax_multiple_k.plot(
            omega_values, approximation_curves[current_K],
            color=curve_colors[index % len(curve_colors)],
            linestyle=curve_styles[index % len(curve_styles)],
            linewidth=2.0,
            label=rf"$K={current_K}$",
        )
    ax_multiple_k.axhline(
        exact_value, color="#222222", linestyle="--", linewidth=1.5,
        label=r"Exact $G_{rs}$", zorder=5,
    )
    ax_multiple_k.set_xlabel(r"Frequency cutoff $\Omega$")
    ax_multiple_k.set_ylabel(r"$G_{rs}$")
    ax_multiple_k.set_title(
        r"$G_{rs}$ approximation for different $K$" + "\n"
        + rf"$n={n},\quad h_r={h_r:g},\quad h_s={h_s:g}$",
        fontsize=11, pad=10,
    )
    ax_multiple_k.set_xlim(0.0, omega_max)
    ax_multiple_k.set_ylim(bottom=0.0)
    ax_multiple_k.grid(axis="y", color="#e6e6e6", linewidth=0.6)
    ax_multiple_k.set_axisbelow(True)
    ax_multiple_k.legend(loc="upper left", frameon=False, ncol=2)
    multiple_k_png_path = image_directory / "g_rs_vs_omega_multiple_k.png"
    fig_multiple_k.savefig(multiple_k_png_path, dpi=200, bbox_inches="tight")
    plt.show()  # Shared exporter automatically saves this figure as PDF.


## Vary K at fixed frequency cutoffs

Place the number of trapezoidal intervals $K$ on the horizontal axis. Each curve holds one cutoff $\Omega$ fixed, while all curves reuse the same sample, bandwidth pair, and exact reference from above. The calculation calls the same [frequency-quadrature helper](../kde_tools/fourier/similarities.py), with `frequency_points=current_K+1`.

The displayed cutoffs are Omega = 5, 10, and 20. The vertical axis is zoomed to the range of these curves and the exact reference.


In [ ]:
# Edit these fixed cutoffs and the horizontal K range.
fixed_omega_values = [5.0, 10.0, 20.0]
K_min = 16
K_max = 160

if not fixed_omega_values or not np.all(
        np.isfinite(fixed_omega_values) & (np.asarray(fixed_omega_values) > 0)):
    raise ValueError("fixed_omega_values must contain finite, positive cutoffs.")
if len(set(fixed_omega_values)) != len(fixed_omega_values):
    raise ValueError("fixed_omega_values must not contain duplicates.")
for value in (K_min, K_max):
    if not isinstance(value, (int, np.integer)) or isinstance(value, bool) or value < 1:
        raise ValueError("K_min and K_max must be positive integers.")
if K_max <= K_min:
    raise ValueError("Require K_min < K_max.")

K_axis = np.arange(K_min, K_max + 1)
curves_vs_K = {}
for current_omega in fixed_omega_values:
    curves_vs_K[current_omega] = np.array([
        gaussian_kde_l2_gram_fourier(
            samples,
            bandwidths,
            frequency_cutoff=float(current_omega),
            frequency_points=int(current_K) + 1,
        )[0, 1]
        for current_K in K_axis
    ])

np.savetxt(
    data_directory / "g_rs_vs_k_multiple_omega.csv",
    np.vstack([
        np.column_stack((
            np.full_like(K_axis, current_omega, dtype=float),
            K_axis,
            curves_vs_K[current_omega],
            np.full_like(K_axis, exact_value, dtype=float),
            np.abs(curves_vs_K[current_omega] - exact_value),
        ))
        for current_omega in fixed_omega_values
    ]),
    delimiter=",",
    header="omega,K,approximation,exact,absolute_error",
    comments="",
)
settings_vs_k = {
    "sample_distribution": settings["sample_distribution"],
    "n": int(n), "seed": int(seed),
    "h_r": float(h_r), "h_s": float(h_s),
    "K_min": int(K_min), "K_max": int(K_max),
    "fixed_omega_values": [float(value) for value in fixed_omega_values],
    "exact_value": exact_value,
    "method_version": FOURIER_L2_METHOD_VERSION,
}
(data_directory / "settings_vs_k.json").write_text(
    json.dumps(settings_vs_k, indent=2) + "\n"
)
print(f"Same fixed sample: n={len(samples)}, exact G_rs={exact_value:.12g}")
print(f"K={K_min},...,{K_max}; fixed cutoffs: {fixed_omega_values}")


In [ ]:
with plt.rc_context({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
}):
    fig_vs_k, ax_vs_k = plt.subplots(figsize=(6.5, 4.0), layout="constrained")
    omega_colors = ["#a455a0", "#e69f00", "#009e73", "#2563a6", "#d55e00"]
    omega_styles = [":", (0, (5, 2, 1, 2)), "-.", "-", "-"]
    for index, current_omega in enumerate(fixed_omega_values):
        ax_vs_k.plot(
            K_axis, curves_vs_K[current_omega],
            color=omega_colors[index % len(omega_colors)],
            linestyle=omega_styles[index % len(omega_styles)],
            linewidth=2.0,
            label=rf"$\Omega={current_omega:g}$",
        )
    ax_vs_k.axhline(
        exact_value, color="#222222", linestyle="--", linewidth=1.5,
        label=r"Exact $G_{rs}$", zorder=5,
    )
    ax_vs_k.set_xlabel(r"Number of trapezoidal intervals $K$")
    ax_vs_k.set_ylabel(r"$G_{rs}$")
    ax_vs_k.set_title(
        r"$G_{rs}$ approximation for fixed frequency cutoffs" + "\n"
        + rf"$n={n},\quad h_r={h_r:g},\quad h_s={h_s:g}$",
        fontsize=11, pad=10,
    )
    ax_vs_k.set_xlim(K_min, K_max)
    from matplotlib.ticker import MaxNLocator
    ax_vs_k.xaxis.set_major_locator(MaxNLocator(nbins=8, integer=True))
    # Fit the vertical scale to the retained curves and the exact reference.
    displayed_values = np.concatenate([
        *[curves_vs_K[value] for value in fixed_omega_values],
        np.array([exact_value]),
    ])
    y_min = float(displayed_values.min())
    y_max = float(displayed_values.max())
    y_padding = max(0.08 * (y_max - y_min), 1e-8 * max(abs(exact_value), 1.0))
    ax_vs_k.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_vs_k.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax_vs_k.ticklabel_format(axis="y", style="plain", useOffset=False)
    ax_vs_k.grid(axis="y", color="#e6e6e6", linewidth=0.6)
    ax_vs_k.set_axisbelow(True)
    ax_vs_k.legend(loc="upper right", frameon=False, ncol=2)
    vs_k_png_path = image_directory / "g_rs_vs_k_multiple_omega.png"
    fig_vs_k.savefig(vs_k_png_path, dpi=200, bbox_inches="tight")
    plt.show()  # Shared exporter automatically saves this figure as PDF.
